# Plantão do Comentário Matinal — com o GitHub Copilot

Esta versão do plantão usa o **GitHub Copilot do VS Code** nas três etapas de IA. Cada
etapa grava a mensagem num arquivo, a célula fica esperando, e você roda o comando da
etapa no chat do Copilot, em **modo agente**: `/matinal-triagem`, `/matinal-redacao` ou
`/matinal-revisao`. O Copilot lê a mensagem, grava a resposta, e a célula continua
sozinha. Abrir o chat: **Ctrl+Alt+I**; o modo fica no seletor embaixo da caixa de texto.

Nenhuma das fachadas implementa o plantão: este notebook, o `plantao.ipynb` e o
`uv run matinal` chamam as mesmas funções de `comentario_matinal.plantao`.

A janela vai das **7h00 às 9h00**, inclusive nas duas pontas, medidas no fuso da
máquina. Fora dela a execução é ensaio: nada é bloqueado, mas o resultado não vai à
diretoria. A faixa da próxima célula diz em qual dos dois estados esta execução está.

**O fechamento do plantão não está aqui.** `uv run matinal enviado` arquiva o
comentário e esvazia `input/comentario_matinal/` e `output/comentario_matinal/`. É o
único passo destrutivo do processo, e notebook é onde se re-executa célula sem querer.
Ele fica no terminal, depois do envio — última célula.

Rodar as células na ordem: cada uma consome o que a anterior gravou em
`output/comentario_matinal/`.

In [ ]:
import os

# Este notebook fala com o GitHub Copilot do VS Code: cada etapa de IA grava a
# mensagem num arquivo, espera você rodar o comando dela no chat do Copilot, em
# modo agente, e lê a resposta que ele grava. Fixado aqui, e não deixado à
# detecção automática, para o notebook se comportar igual em qualquer máquina.
os.environ["COMENTARIO_MATINAL_BACKEND"] = "copilot"

%matplotlib inline

from html import escape

from IPython.display import HTML, Markdown, display

from comentario_matinal import plantao
from comentario_matinal.janela import faixa

# O núcleo diz o que falta e para aí. A frase que ensina a suprir é de cada
# fachada: no terminal ela manda rodar `uv run matinal`, e aqui isso seria
# conselho errado — não há linha de comando alguma na célula que você está
# olhando. Estas são as frases deste notebook.
REMEDIO = {
    plantao.COLETA_AUSENTE: " Rodar as células do Passo 1 antes desta.",
    plantao.TRIAGEM_AUSENTE: " Rodar a célula da triagem, no Passo 2.",
    plantao.TRIAGEM_ILEGIVEL:
        " Escrever os temas à mão na célula do Passo 3, em vez de escolhê-los por número.",
    plantao.REDACAO_AUSENTE: " Rodar a célula da redação, no Passo 4.",
    plantao.TEMAS_AUSENTES: " Preencher a célula TEMAS, no Passo 3, e rodá-la.",
    # Os códigos de `--forcar` não aparecem aqui: o fechamento do plantão é o
    # único passo que não existe neste notebook, e por isso as recusas dele
    # também não.
}

# O mesmo fato dito noutra língua: para o comando o horário fixado é `--asof`,
# e aqui é um argumento de `plantao.contexto()`.
AVISO = {
    plantao.ASOF_DIVERGE_DO_PAINEL:
        "Aviso: o asof passado a plantao.contexto() ({asof}) diverge da "
        "referência do painel ({painel}). O painel é o material que a etapa "
        "analisa; conferir se é mesmo o do dia.",
}

FAIXA = ('<div style="background:{cor};color:#fff;padding:12px 16px;'
         'border-radius:6px;font-size:15px;{extra}">{texto}</div>')


def mostra(mensagem):
    """O `progresso` das células: cada aviso aparece enquanto o passo roda.

    A chamada ao modelo leva minutos, e aviso que chega depois da espera já não
    serve para decidir se valia a pena interromper. Não há aqui a memória que o
    `_Voz` do comando mantém, porque o notebook não precisa dela: o que já foi dito
    continua visível logo acima, na saída da própria célula.
    """
    codigo = getattr(mensagem, "codigo", None)
    if codigo in AVISO:
        mensagem = AVISO[codigo].format(**mensagem.dados)
    print(mensagem, flush=True)


class ParadaDoAutor(Exception):
    """A parada que o Run All respeita — a decisão do Passo 3, não uma falha.

    Fica aqui, e não em `plantao`, porque o terminal não precisa dela: lá a
    fronteira já existe, entre `matinal triagem` e `matinal redacao`, e quem a
    atravessa é uma pessoa digitando o segundo comando. O notebook não tem essa
    fronteira — "executar tudo" é um item de menu —, e é ela que esta classe repõe.
    """


def parada(texto):
    """Interrompe a execução em cadeia, deixando a célula marcada como erro.

    É a marca de erro que faz o front-end abortar as células ainda na fila: vale
    no JupyterLab, no VS Code e no `nbconvert --execute`. A faixa de `_explica` é
    só o que o autor vê no lugar do traceback; quem para o Run All é a exceção.
    """
    raise ParadaDoAutor(texto)


def _explica(shell, tipo, erro, tb, tb_offset=None):
    """Mostra a parada do plantão como faixa, e não como traceback.

    Vale para as duas paradas, e a cor as separa: vermelho é falha, âmbar é o
    processo funcionando. Pintar a decisão do Passo 3 de vermelho ensinaria a
    ignorar vermelho, que é a única cor deste notebook que precisa ser lida.

    `ErroDePlantao` é falha prevista — falta um insumo, falta uma decisão —, e o
    traceback só empurraria para longe da tela a única linha que interessa. Os
    avisos não são repetidos: o `progresso` já os mostrou logo acima, na saída
    desta mesma célula.
    """
    linhas = [f"{erro}{REMEDIO.get(getattr(erro, 'remedio', None), '')}"]
    destino = getattr(erro, "destino", None)
    if destino:
        linhas.append(f"A resposta completa do modelo está em {destino}.")
    cor = "#8a5a00" if isinstance(erro, ParadaDoAutor) else "#b00020"
    display(HTML(FAIXA.format(cor=cor, extra="white-space:pre-wrap",
                              texto=escape("\n".join(linhas)))))


get_ipython().set_custom_exc((plantao.ErroDePlantao, ParadaDoAutor), _explica)

ctx = plantao.contexto()

# No terminal o banner de dry run interrompe a leitura no stderr; numa célula uma
# linha de log passaria batido. Por isso a faixa — e por isso ela também aparece no
# caso normal: faixa que só existe no erro ensina a ignorar o espaço onde ela
# apareceria.
if ctx.dry_run:
    display(HTML(FAIXA.format(
        cor="#b00020", extra="font-weight:600",
        texto=f"DRY RUN — fora da janela de {faixa()}. "
              "Execução de ensaio; não enviar o resultado à diretoria.")))
else:
    display(HTML(FAIXA.format(
        cor="#0b6e3a", extra="font-weight:600",
        texto=f"Plantão — dentro da janela de {faixa()}.")))

for aviso in ctx.avisos:
    # O dry run acabou de sair como faixa; repeti-lo em texto logo abaixo diria
    # a mesma frase duas vezes. Os demais avisos do contexto — divergência de
    # fuso — não têm outro lugar por onde sair.
    if getattr(aviso, "codigo", None) != plantao.DRY_RUN:
        mostra(aviso)

## Passo 1 — Coleta

É o único passo que toca a Bloomberg. Exige o terminal **aberto e logado**, e rodando
do Windows nativo, nunca do WSL: o `blpapi` conversa com o terminal por IPC local.

A coleta é única e serve às três saídas — a imagem colada no e-mail, o texto que as
etapas de IA leem como estado do mercado e o documento final. Por construção elas
descrevem os mesmos números.

Reunir antes as fontes do dia em `input/comentario_matinal/`, **só em PDF**: as etapas
recebem texto, nunca anexo nem imagem. É a `input/` da **raiz do repositório** — ao
lado de `prompts/` —, não uma pasta dentro de `notebooks/`: os caminhos padrão são
ancorados na raiz, em `config.py`, e uma `notebooks/…/input/` faria a etapa avisar
"nenhum PDF aproveitado" sem dizer por quê.

Sem licença BQL na máquina, a consulta do calendário estoura. A célula do calendário
tem o `SEM_CALENDARIO` para esse caso — é o equivalente ao `--sem-calendario` do
terminal.

**O comentário do dia anterior é opcional.** Salve o e-mail enviado ontem em PDF,
com nome começando por `anterior` — `anterior*.pdf`, por exemplo
`anterior_2026-09-28.pdf` —, junto das fontes: ele vai para o campo próprio, não é
lido como notícia do dia, e vale mesmo que esta máquina tenha um comentário
arquivado, que pode ser de dias antes. Sem o PDF, entra o arquivado mais recente da
semana, se houver; sem nenhum dos dois, as etapas trabalham só com as fontes.

In [ ]:
mercado = plantao.coleta_mercado(ctx, progresso=mostra)

print(f"{len(mercado.referencia)} ativo(s) com referência; "
      f"{len(mercado.indisponiveis)} sem.")
mercado.referencia.head()

In [ ]:
painel = plantao.desenha_painel(ctx, mercado)

print(f"Painel:     {painel.caminho}")
painel.figura

In [ ]:
# Numa máquina sem licença BQL a consulta do calendário estoura. Trocar para True
# pula o passo, como o `--sem-calendario` do terminal: o bloco direcional sai sem a
# agenda — e diz isso no próprio texto —, e o Passo 7 passa a recusar a montagem,
# porque a tabela é uma das duas imagens que o template espera.
SEM_CALENDARIO = False

calend = None
if not SEM_CALENDARIO:
    calend = plantao.prepara_calendario(ctx, progresso=mostra)

    # Os avisos do calendário nascem depois da consulta, e não passam por `progresso`.
    for aviso in calend.avisos:
        mostra(aviso)

    print(f"Calendário: {calend.caminho_md}")
else:
    print("Calendário pulado (SEM_CALENDARIO). As etapas rodam sem a agenda do dia.")

calend.eco if calend is not None else None

In [ ]:
bloco = plantao.monta_bloco(ctx, mercado, painel, calend)

for aviso in bloco.avisos:
    mostra(aviso)

print(f"Texto:      {bloco.caminho}\n")
print(bloco.texto)

## Passo 2 — Triagem

A primeira das três etapas de IA. Nenhuma delas toca a Bloomberg: todas consomem o
que o Passo 1 gravou em `output/comentario_matinal/`.

Antes de rodar, conferir que os PDFs do dia estão em `input/comentario_matinal/` — wraps da Bloomberg,
First Word, e-mails de sell-side, matérias do FT ou do WSJ. Arquivo que não seja PDF
é ignorado, e a etapa avisa qual foi.

O `progresso=mostra` existe para esta espera: a chamada ao modelo leva minutos, e os
avisos de "sem fontes noticiosas" ou "sem o comentário do dia anterior" só servem
enquanto ainda vale a pena interromper. Eles continuam no `.avisos` do resultado.

**No Copilot.** Quando a célula mostrar "No chat do Copilot … digite
/matinal-triagem", abrir o chat (Ctrl+Alt+I), escolher o modo **Agente**, digitar
`/matinal-triagem` e enviar. O Copilot lê a mensagem — é longa, e ele a lê em
trechos — e grava a resposta; a célula percebe e continua. Se o VS Code pedir para
confirmar a gravação do arquivo, confirmar. Se a célula acusar "não foi lida até o
fim" ou "código de outra execução", rodar a célula de novo e repetir o comando. A
espera dura até 15 minutos; interromper o kernel a cancela.

In [ ]:
# `web` fica False: o prompt file do Copilot só lê e grava arquivos, e tudo o
# que a etapa precisa vai injetado na mensagem.
triagem = plantao.roda_etapa(ctx, "triagem", web=False, progresso=mostra)

print(f"\nTriagem:    {triagem.caminho}\n")
display(Markdown(triagem.texto))

## Passo 3 — A decisão é sua

Ler a triagem acima e escolher os temas, **o dominante primeiro**. É o único ponto do
processo em que a decisão é editorial e humana; o comando e o notebook a pedem de
jeitos diferentes, mas nenhum dos dois a toma.

A célula seguinte interrompe a execução de propósito: um **Run All** para nela, antes
de a escolha ser usada, e a faixa âmbar que aparece não é erro. Editar o `ESCOLHA` na
célula logo abaixo dela — os números da triagem, o dominante primeiro — e seguir dali
para baixo.

In [ ]:
# Run All para aqui, de propósito. A escolha dos temas é a única decisão que o
# processo não toma sozinho, e uma execução corrida passaria por ela levando a
# lista que ficou na célula desde ontem — para só então gastar minutos de modelo
# redigindo sobre ela. No terminal esta parada é de graça: são dois comandos.
parada("Passo 3 — ler a triagem acima e editar o ESCOLHA na célula abaixo, o "
       "tema dominante primeiro. Seguir a partir dela.")

In [ ]:
# Os números da tabela de temas candidatos que a triagem acabou de produzir,
# dominante primeiro. O texto sai da própria triagem: reescrevê-lo à mão é
# transcrição, e transcrição erra.
ESCOLHA = [1, 2, 3, 4, 5]

TEMAS = plantao.temas_da_triagem(ctx, ESCOLHA)

# A ressalva é o que você acrescenta e a tabela não tem como saber: um limite
# temporal, uma atribuição obrigatória, uma direção que o painel contradiz.
# Em 17/08 foi uma ressalva assim que impediu o comentário de afirmar que o
# dólar caíra na sessão, quando o painel mostrava o câmbio estável.
#
# TEMAS += (
#     "\n- Ressalva: as moedas estão estáveis na sessão corrente; "
#     "não descrever o movimento como queda de hoje."
# )

print(TEMAS)

## Passo 4 — Redação

Recebe os temas da célula acima e os alertas da triagem, extraídos da seção C do
arquivo da etapa anterior — não é preciso copiá-los à mão.

Sai o comentário com o bloco de auditoria. Conferir ali a contagem total, o orçamento
por marcador, o mapeamento marcador → fonte e as ressalvas.

**No Copilot:** quando a célula pedir, `/matinal-redacao`, em modo agente.

In [ ]:
redacao = plantao.roda_etapa(ctx, "redacao", temas=TEMAS, progresso=mostra)

print(f"\nRedacao:    {redacao.caminho}\n")
display(Markdown(redacao.texto))

## Passos 5 e 6 — Passagem ao revisor e revisão

A passagem (Passo 5) é humana e acontece fora daqui: vão ao segundo analista o texto,
os PDFs das fontes, o painel em texto, o bloco de auditoria e **o horário de redação**.
A folga é de pelo menos dez minutos sobre o horário de envio pretendido.

A revisão (Passo 6) lê a redação acima e o comentário do dia anterior — é dele que sai
a checagem de contradição entre um dia e o seguinte. É a única etapa cuja saída entra
direto no documento que vai à diretoria: o texto revisado é extraído para
`output/comentario_matinal/comentario_AAAAMMDD.md`, e formato inesperado interrompe em vez de gravar.

**No Copilot:** quando a célula pedir, `/matinal-revisao`, em modo agente.

In [ ]:
revisao = plantao.roda_etapa(ctx, "revisao", progresso=mostra)

print(f"\nRevisao:    {revisao.caminho}")
print(f"Comentário: {revisao.comentario}\n")
display(Markdown(revisao.texto))

## Passo 7 — Montagem do documento

O `.docx` sai do template com as imagens **da data**, não com as de agora: o
comentário foi escrito contra o painel gravado, e recoletar aqui produziria um
documento cujo texto e cuja imagem descrevem manhãs diferentes.

Sem a tabela do calendário a montagem é recusada — o template tem dois lugares de
imagem, e um deles sairia vazio no documento que vai à diretoria. É o que acontece
quando o Passo 1 rodou com `SEM_CALENDARIO = True`.

Depois de montado, abrir no Word para inserir o gráfico do dia, se houver, conferir o
texto e exportar o PDF.

In [ ]:
docx = plantao.monta_documento(ctx, revisao.comentario)

print(f"Documento:  {docx}")

## Passo 8 — Conferência, antes do e-mail

Rodar **depois** de editar o `.docx` no Word e **antes** de mandar o e-mail: é a única
janela em que uma divergência ainda tem conserto.

O que fica arquivado é o `.md`, e é ele que a triagem de amanhã lê como comentário do
dia anterior. Uma correção feita só no Word desaparece do processo no dia seguinte.

In [ ]:
from comentario_matinal.enviado import relatorio_da_conferencia

divergencias = plantao.confere(ctx)

# O relatório é o mesmo do terminal, palavra por palavra: ele tem um dono só, no
# núcleo, e cada fachada escolhe apenas onde mostrar as linhas.
for linha in relatorio_da_conferencia(divergencias, ctx.marca):
    print(linha)

## Passo 9 — Depois do envio, no terminal

```
uv run matinal enviado
```

Este passo **não** existe aqui, de propósito. Ele arquiva o `.md` em `arquivo/comentario_matinal/` — de
onde a triagem de amanhã o lê como comentário do dia anterior — e em seguida esvazia
`input/comentario_matinal/` e `output/comentario_matinal/`. É o único passo destrutivo do processo, e o único que afirma um
fato que nenhum código pode verificar: que o e-mail foi mesmo enviado.

Numa célula, uma re-execução distraída não custaria nada; aqui custaria o dia. Rodar
no terminal, depois que a diretoria recebeu o comentário.

Antes de fechar: **limpar as saídas deste notebook** (Kernel → Restart & Clear Output).
As células executadas carregam dados de mercado e o texto do comentário. O filtro do
git as tira no `git add`, se tiver sido instalado com `uv run nbstripout --install` —
mas isso não é motivo para deixá-las aí.